In [0]:
import sys, os
from getpass import getpass
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import col, regexp_replace, when

# Garante que o Python enxergue o arquivo spark_session.py
sys.path.append(os.path.abspath(".."))
from spark_session import create_spark_session

spark = create_spark_session()

print("📌 Conexão com Postgres Neon - Iniciando carregamento para Landing...\n")

# Configs do DB
host = "spark-delta-iceberg-wsl.c0fk60kqcz2v.us-east-1.rds.amazonaws.com"
porta = "5432"
database = "postgres"
usuario = "postgres"
senha = "postgres"

jdbc_url = f"jdbc:postgresql://{host}:{porta}/{database}"

db_properties = {
    "user": usuario,
    "password": senha,
    "driver": "org.postgresql.Driver"
}

# Lendo os dados necessário p/ dps exportar para a landing
tabela_origem = "products_raw"
print(f"🔄 Lendo dados de '{tabela_origem}' do Postgres...")

df = spark.read.jdbc(url=jdbc_url, table=tabela_origem, properties=db_properties)
print("✅ Dados carregados do Postgres (raw):")
df.show(truncate=False)

# Limpe e conversão das colunas
df_clean = df \
    .withColumn("quantidade", when(col("quantidade").rlike("^[0-9]+$"), col("quantidade").cast(IntegerType())).otherwise(F.lit(None).cast(IntegerType()))) \
    .withColumn("valor", when(regexp_replace("valor", ",", ".").rlike("^[0-9]+(\\.[0-9]+)?$"), regexp_replace("valor", ",", ".").cast(DoubleType())).otherwise(F.lit(None).cast(DoubleType()))) \
    .withColumn("estado", F.trim(F.upper(col("estado")))) \
    .withColumn("ingest_timestamp", col("ingest_timestamp").cast(TimestampType()))

# Adição de metadados na landing
df_landing = df_clean.withColumn("data_carga", F.current_timestamp())

# Salva os dados em formato delta
nome_tabela_landing = "landing_products"
df_landing.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_landing)

print(f"\n🚀 Finalizado! Dados de {tabela_origem} → landing salva como Delta Table: {nome_tabela_landing}\n")
print("Use no Databricks:")
print(f"SELECT * FROM {nome_tabela_landing} LIMIT 10;")
